# 1.기본 손가락 추적

- 손가락 랜드마크 확인 : https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker?hl=ko

In [2]:
# OpenCV: 웹캠 입력/색상 변환/화면 출력
import cv2
# MediaPipe: 이미지 래핑/모델 실행
import mediapipe as mp

In [1]:
# HandLandmarker 준비
from pathlib import Path
import urllib.request

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [3]:
# 모델 저장 폴더
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# 손 위치/21개 랜드마크 검출 모델
HAND_LANDMARKER_MODEL = MODEL_DIR / "hand_landmarker.task"
HAND_LANDMARKER_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"

In [4]:
def download_model(model_path, model_url):
    # 기존 모델은 다운로드 생략
    if model_path.exists():
        return

    # 모델이 없을 때만 다운로드
    print(f"모델 다운로드 중: {model_path}")
    urllib.request.urlretrieve(model_url, model_path)


# 첫 실행 시 손 랜드마크 모델 다운로드
download_model(HAND_LANDMARKER_MODEL, HAND_LANDMARKER_MODEL_URL)

모델 다운로드 중: models\hand_landmarker.task


In [5]:
# Tasks API 클래스 별칭
BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions
mp_hands = vision.HandLandmarksConnections
mp_drawing = vision.drawing_utils

In [7]:
def create_hand_landmarker(num_hands=2):
    # 손 랜드마크 검출기 생성
    options = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(HAND_LANDMARKER_MODEL)),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=num_hands,
        min_hand_detection_confidence=0.5, # 손이 잘 안 잡히면 낮추고, 배경을 손으로 잘못 잡으면 높인다.
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )
    return HandLandmarker.create_from_options(options)


def draw_hand_landmarks(image, hand_result):
    # 손마다 21개 랜드마크 표시
    for hand_landmarks in hand_result.hand_landmarks:
        mp_drawing.draw_landmarks(
            image,
            hand_landmarks,                 # 검출된 손 랜드마크 
            mp_hands.HAND_CONNECTIONS,      # 손가락 관절 연결 정보
            mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),  # 점 스타일
            mp_drawing.DrawingSpec(color=(250, 44, 250), thickness=2)   # 선 스타일
        )

In [9]:
# 0번 카메라 열기
cap = cv2.VideoCapture(0)

# FPS 없으면 30 사용
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

# 손 2개까지 추적
with create_hand_landmarker(num_hands=2) as hand_landmarker:
    while cap.isOpened():
        # 웹캠 프레임 읽기
        success, image = cap.read()
        if not success:
            continue

        # BGR -> RGB 변환
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        # 프레임 timestamp 계산
        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        # 현재 프레임 손 랜드마크 검출
        result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

        # 검출된 손의 점과 연결선을 이미지 위에 그림
        draw_hand_landmarks(image, result)

        cv2.imshow('Hand Tracking', image)

        # q 입력 시 종료
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

# 2. 손가락 포즈 검출

- 바위 : 모든 손가락이 접혀 있음
- 가위 : 엄지를 제외한 두 손가락(검지와 중지)만 펴져 있음
- 보 : 모든 손가락이 펴져 있음

In [ ]:
# 한글 출력용 Pillow 설치 명령
# 이미 설치되어 있으면 생략 가능
# %pip install Pillow

In [10]:
# OpenCV 한글 출력 보완용 PIL
from PIL import ImageFont, ImageDraw, Image
import numpy as np


def draw_text(img, text, position, font_size, font_color):
    # Windows 기본 한글 폰트 경로. 다른 OS에서는 폰트 경로를 바꿔야 함.
    font_path = "C:/Windows/Fonts/gulim.ttc"

    # opencv 이미지를 PIL이미지로 변환
    img_pil = Image.fromarray(img)

    # PIL Draw 객체 생성
    draw = ImageDraw.Draw(img_pil)

    # 폰트 스타일 지정
    font = ImageFont.truetype(font_path, font_size)

    # position 위치에 text를 그림. font_color는 RGB 순서.
    draw.text(position, text, font=font, fill=font_color)

    # 다시 OpenCV에서 사용할 수 있도록 최종 numpy array 로 이미지 형태 반환.
    return np.array(img_pil)

In [11]:
def get_hand_pose(hand_landmarks, handedness=None):
    # MediaPipe 버전에 맞는 랜드마크 리스트 선택
    landmarks = hand_landmarks.landmark if hasattr(hand_landmarks,'landmark') else hand_landmarks

    open_fingers = []

    # 엄지 랜드마크 선택
    thumb_tip = landmarks[4]
    thumb_ip = landmarks[3]
    thumb_mcp = landmarks[2]
    pinky_mcp = landmarks[17]

    # 엄지 펼침 판정
    if thumb_mcp.x < pinky_mcp.x:
        open_fingers.append(thumb_tip.x < thumb_ip.x)
    else:
        open_fingers.append(thumb_tip.x > thumb_ip.x)

    # 검지~새끼 펼침 판정
    # Mediapipe 이미지 좌표계에서는 위쪽으로 갈수록 y값이 작아짐
    # 손끝 Tip의 y값이 Pip 관절보다 작으면 손가락이 위로 펴져있다.
    for tip_idx, pip_idx in [(8, 6), (12, 10), (16, 14), (20, 18)]:
        tip = landmarks[tip_idx]
        pip = landmarks[pip_idx]
        open_fingers.append(tip.y < pip.y)

    thumb_open, index_open, middle_open, ring_open, pinky_open = open_fingers
    
    # 펴진 손가락 개수를 셈
    open_count = open_fingers.count(True)


    # 가위/바위/보 판정
    if open_count == 0:
        return "바위"
    elif open_count == 5:
        return "보"
    elif open_count == 2 and index_open:
        return "가위"
    else:
        return "모름"

In [13]:
# 포즈 판별용 웹캠 루프
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

# 가위/바위/보는 한 손 기준
with create_hand_landmarker(num_hands=1) as hand_landmarker:
    while cap.isOpened():
        success, image = cap.read()
        if not success:
            continue

        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        # 손 랜드마크/handedness 검출
        result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.hand_landmarks:
            for idx, hand_landmarks in enumerate(result.hand_landmarks):
                # 손 관절 화면 표시
                mp_drawing.draw_landmarks(
                    image,
                    hand_landmarks,                     # 검출된 손 랜드마크
                    mp_hands.HAND_CONNECTIONS,          # 손가락 관절 연결 정보
                    mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),  # 점 스타일
                    mp_drawing.DrawingSpec(color=(250, 44, 250), thickness=2) # 선 스타일
                )
                
                # 왼손/오른손 분류 결과
                # 손바닥을 카메라로 향했을 때 오른손 -> 엄지가 화면 왼쪽 / 왼손 -> 엄지가 화면 오른쪽
                handedness = result.handedness[idx] if idx < len(result.handedness) else None

                # 손가락 상태 -> 가위/바위/보 변환
                pose = get_hand_pose(hand_landmarks, handedness)

                # 한글 결과 출력
                image = draw_text(image, pose, (10, 50), 30, (255, 255, 2555))

        cv2.imshow('Hand Pose', image)

        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()